In [ ]:
# 安裝必要套件（Colab 環境）
!pip install transformers torch scikit-learn pandas matplotlib seaborn ckip_transformers -q
!pip install -U ehownet

print("套件安裝完成！")

In [ ]:
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, f1_score
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from collections import Counter
import warnings

warnings.filterwarnings('ignore')

#中文
# 1. 安裝中文字型（Colab 用）
!apt-get install -y fonts-noto-cjk

# 2. 清除快取（保留這步）
!rm -rf ~/.cache/matplotlib

# 3. 重新載入 font manager（正確方法）
from matplotlib import font_manager
font_manager.fontManager.addfont('/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc')

# 4. 設定字型
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK JP']
plt.rcParams['axes.unicode_minus'] = False

sns.set(font='Noto Sans CJK JP', rc={'axes.unicode_minus': False})

print("所有套件載入完成！")
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")

In [ ]:
import torch
from ckip_transformers.nlp import CkipWordSegmenter, CkipPosTagger

# 自動判斷：如果有 GPU 則 device=0，否則 device=-1 (CPU)
device_id = 0 if torch.cuda.is_available() else -1
print(f"目前使用的裝置編號: {device_id} ({'GPU' if device_id == 0 else 'CPU'})")

# 初始化斷詞器與詞性標記器
ws_driver = CkipWordSegmenter(model="albert-base", device=device_id)
pos_driver = CkipPosTagger(model="albert-base", device=device_id)

print("CKIP 載入完成！")



---



In [ ]:
# ============================================================
# 超參數設定
# ============================================================
MODEL_NAME = "ckiplab/bert-base-chinese"
MAX_LEN = 512
BATCH_SIZE = 8    # 資料集較小（1000筆），使用小 batch size
EPOCHS = 10       # 資料少，多訓練幾個 epoch
LR = 2e-5

Gamma=2
Alpha=0.2
Lambda=0.1


# 四個預測欄位及其標籤（依照競賽官方規格定義）
EVAL_FIELDS = {
    "promise_status": ["Yes", "No"],
    "verification_timeline": ["already", "within_2_years", "between_2_and_5_years", "more_than_5_years", "N/A"],
    "evidence_status": ["Yes", "No", "N/A"],
    "evidence_quality": ["Clear", "Not Clear", "Misleading", "N/A"]
}

# 各欄位的評分權重
FIELD_WEIGHTS = {
    "promise_status": 0.20,
    "evidence_status": 0.30,
    "evidence_quality": 0.35,
    "verification_timeline": 0.15,
}

# 建立標籤 ↔ ID 的對應表
label2id = {field: {lab: i for i, lab in enumerate(labels)} for field, labels in EVAL_FIELDS.items()}
id2label = {field: {i: lab for i, lab in enumerate(labels)} for field, labels in EVAL_FIELDS.items()}
num_labels = {field: len(labels) for field, labels in EVAL_FIELDS.items()}

print("超參數設定完成！")
print("\n標籤對應表：")
for field, mapping in label2id.items():
    print(f"  {field}: {mapping}")

In [ ]:
import re
from sklearn.model_selection import train_test_split
import json
import os

# 定義檔案路徑
TRAIN_FILE = "vpesg4k_train_1000.json"
VAL_FILE = "vpesg4k_val_1000.json"

def load_json(file_path):
    # 檢查檔案是否存在
    if not os.path.exists(file_path):
        print(f"❌ 錯誤：找不到檔案 {file_path}！")
        return None

    # 讀取 JSON
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"✅ {file_path} 載入成功，共 {len(data)} 筆資料")
    return data

# 分別讀取
train_data = load_json(TRAIN_FILE)
val_data = load_json(VAL_FILE)


# 檢查是否成功
if train_data is not None and val_data is not None:
    print("🎉 所有資料載入完成！")



In [ ]:
from ckip_transformers.nlp import CkipWordSegmenter
# --- 1. 定義處理函式 (包含斷詞與過濾) ---
def batch_ckip(texts, batch_size=32):
    all_ws, all_pos = [], []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        ws = ws_driver(batch)
        pos = pos_driver(ws)
        all_ws.extend(ws)
        all_pos.extend(pos)
    return all_ws, all_pos
def process_data_segmentation(data_list):
    # 提取文字
    texts = [item["data"] for item in data_list]
    # 批次斷詞
    ws_results, pos_results = batch_ckip(texts, batch_size=32)

    keep_pos_prefixes = {'N', 'V'}

    for i, (words, pos) in enumerate(zip(ws_results, pos_results)):
        filtered = [w for w, p in zip(words, pos)
                    if (p[0] in keep_pos_prefixes or p == "FW") and "CATEGORY" not in p]
        # 存回 dict
        data_list[i]['segmented_text'] = " ".join(filtered)
    return data_list

# --- 2. 執行處理 ---
print("正在處理訓練集...")
train_data = process_data_segmentation(train_data)

print("正在處理驗證集...")
val_data = process_data_segmentation(val_data)

print("✅ 預處理完成！")

# --- 3. 檢查結果 ---
print(f"訓練集範例: {train_data[0].get('segmented_text', '找不到欄位')[:100]}...")
print(f"驗證集範例: {val_data[0].get('segmented_text', '找不到欄位')[:100]}...")



---



In [ ]:
KEYWORDS_LIB = {
    'T1': {
        'promise': {'承諾', '保證', '保障', '確保', '允諾', '擔保', '保証', '許諾',
              '致力', '推動', '促進', '推展', '推行', '執行', '開展',
              '確立', '訂立', '建立', '制定', '設定',
              '秉持', '堅持', '遵循', '遵守', '落實', '執行', '實踐',
              '持續', '長期', '穩定', '永續', '持久',
              '計畫', '規劃', '宣示', '接軌', '納入', '協助', '改善', '制定',
              '因應', '應對', '應變', '採取措施',
              '攜手', '未來', '展望', '目標', '願景',
              'ISO', 'TCFD', 'ESG', 'GRI'},
        'esg_env': {'碳', '碳排', '排放', '碳足跡', '中和', '減碳', '減排',
              '環境', '保護', '環保', '轉型',
              '再生', '回收', '能源', '綠電', '循環', '經濟', '永續', '綠能', '清潔', '節能', '節電',
              '生物', '生態', '生態系', '多樣性',
              '氣候', '變遷', '暖化', '污染',
              '韌性', '採購', '範疇'},
        'esg_soc': {'員工', '性別', '平等', '公平', '多元', 'DEI',
              '管理', '人才', '發展', '培育', '招募', '教育', '訓練', '學習', '留才', '離職率',
              '社會', '責任', 'CSR',
              '人權', '政策', '保障', '勞動', '勞工', '童工',
              '福利', '關懷', '滿意度', '敬業度', '勞資',
              '薪資', '薪酬', '工時', '條件',
              '職場', '職安', '職災', '職涯',
              '包容', '平權', '歧視', '性騷擾',
              '公益', '志工', '志願'},
        'esg_gov': {'董事會', '理事會', '監察人', '委員會', '公司',
              '合規', '法遵', '規範', '法規', '法令',
              '透明', '揭露', '公開',
              '風險', '風控', '評估', '控管', '監控', '內控', '管控', '控制',
              '誠信', '道德', '廉潔', '經營', '倫理',
              '經營', '營運', '管理', '策略', '治理',
              '貪腐', '賄賂', '舞弊',
              '審計', '稽核', '查核',
              '資安', '安全', '個資', '隱私',
              '薪酬', '薪資', '報酬', '獎酬', '制度', '機制'}
    },
    'T2_T3': {
        'clear': {'完成', '達到', '達成', '實現',
              '建立', '設立', '成立', '制定',
              '通過', '批准', '認可', '取得', '獲得',
              '削減', '降低', '減少', '減緩', '節省',
              '符合', '遵循', '遵守', '依循',
              '辦理', '執行', '實施', '推行', '運作',
              '落實', '實行', '實踐', '導入', '應用',
              '改善', '優化', '提升', '精進', '改進',
              '汰舊換新', '更新', '升級', '換代',
              '稽核', '審核', '查核', '檢查',
              '編列', '規劃', '安排', '列入',
              '認證', '證明', '核可', '驗證',
              '採用', '使用', '運用', '導入',
              '轉型', '改造',
              '%'},
        'not_clear': {'計畫', '規劃', '安排', '設計', '籌劃',
                '期盼', '期望', '希望', '願景',
                '呼應', '響應', '配合', '回應',
                '內部', '內控','自我', '自我', '自主',
                '應', '應對', '採取', '應用',
                '考量', '考慮', '評估', '權衡',
                '攜手','致力'},
        'misleading': {'領先', '最', '全方位', '世界級', '極致'}
    },
    'T4': {
        'already': {'2023', '2024', '去年', '當年度', '已', '目前', '完成', '達成', '以往', '歷年', '累計', '前一年度','過去'},
        'within_2y': {'2026', '2027', '明年', '下年度', '即將', '近兩年', '短期', '近期', '首要目標'},
        'between_2_5y': {'2029', '2030', '中期', '階段性', '中程'},
        'above_5y': {'2040', '2050' , '長期', '遠期','終極目標','持續'}
    }
}

In [ ]:
SEMANTIC_CLASSES = {
    'promise': 0,
    'evidence': 1,
    'quality_score': 2,
    'timeline_score': 3
}

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vec   = TfidfVectorizer(max_features=5000)
train_texts = [item["segmented_text"] for item in train_data]
val_texts   = [item["segmented_text"] for item in val_data]

tfidf_train = tfidf_vec.fit_transform(train_texts)   # fit + transform 訓練集
tfidf_val   = tfidf_vec.transform(val_texts)          # 只 transform 驗證集
word_to_idx = tfidf_vec.vocabulary_
print(f"✅ TF-IDF 建立完成：訓練集 {tfidf_train.shape}, 驗證集 {tfidf_val.shape}")

In [ ]:
def extract_semantic_features(
    text,
    text_idx,
    keywords_lib,
    semantic_classes,
    tfidf_matrix,
    word_to_idx,
    debug=False
):
    vec = [0.0] * len(semantic_classes)

    # sparse row（避免 toarray）
    sample_vec = tfidf_matrix[text_idx]

    def get_tfidf_sum(word_set):
        total = 0.0
        for word in word_set:
            idx = word_to_idx.get(word)
            if idx is not None:
                total += sample_vec[0, idx]
        return total

    # =========================
    # 1. Promise
    # =========================
    t1_score = 0
    promise_t = get_tfidf_sum(keywords_lib['T1']['promise'])
    esg_t = get_tfidf_sum(
        keywords_lib['T1']['esg_env'] |
        keywords_lib['T1']['esg_soc'] |
        keywords_lib['T1']['esg_gov']
    )

    promise_c = sum(w in text for w in keywords_lib['T1']['promise'])
    esg_c = sum(w in text for w in (
      keywords_lib['T1']['esg_env'] |
      keywords_lib['T1']['esg_soc'] |
      keywords_lib['T1']['esg_gov']
    ))

    if promise_c>=1 and esg_c>=1:
        t1_score = (
          promise_c* promise_t
          +esg_c*esg_t
        )
    if debug:
      print("=== T1 ===")
      print("promise_t:", promise_t, "count:", promise_c, "contrib:", promise_c * promise_t)
      print("esg_t:", esg_t, "count:", esg_c, "contrib:", esg_c * esg_t)
      print("t1_score:", t1_score)

    vec[semantic_classes['promise']] = min(t1_score, 2.0)

    # =========================
    # 2. Evidence
    # =========================
    t2_score = 0

    evidence_t = get_tfidf_sum(
        keywords_lib['T2_T3']['clear'] |
        keywords_lib['T2_T3']['not_clear'] |
        keywords_lib['T2_T3']['misleading']
    )
    t2_all = (keywords_lib['T2_T3']['clear'] |
              keywords_lib['T2_T3']['not_clear'] |
              keywords_lib['T2_T3']['misleading']
              )
    evidence_c = sum(1 for w in t2_all if w in text)
    t2_score = (evidence_c*evidence_t)
    if debug:
      print("=== T2 ===")
      print("evidence_t:",evidence_t)
      print("evidence_c:", evidence_c)
      print("t2_score:", t2_score)

    vec[semantic_classes['evidence']] = min(t2_score, 2.0)

    # =========================
    # 3. Quality score
    # =========================
    t3_score = 0

    clear_t = get_tfidf_sum(keywords_lib['T2_T3']['clear'])
    not_clear_t = get_tfidf_sum(keywords_lib['T2_T3']['not_clear'])
    mis_t = get_tfidf_sum(keywords_lib['T2_T3']['misleading'])

    clear_c = sum(1 for w in keywords_lib['T2_T3']['clear'] if w in text)
    not_clear_c = sum(1 for w in keywords_lib['T2_T3']['not_clear'] if w in text)
    misleading_c = sum(1 for w in keywords_lib['T2_T3']['misleading'] if w in text)

    t3_score = (
      clear_c * clear_t
      - not_clear_c * not_clear_t
      - misleading_c* mis_t
    )
    if debug:
      print("=== T3 ===")
      print("clear_t:", clear_t, "count:", clear_c, "contrib:", clear_c * clear_t)
      print("not_clear_t:", not_clear_t, "count:", not_clear_c, "contrib:", -not_clear_c * not_clear_t)
      print("mis_t:", mis_t, "count:", misleading_c, "contrib:", -misleading_c * mis_t)
      print("t3_score:", t3_score)
    vec[semantic_classes['quality_score']] = max(min(t3_score, 2.0), -2.0)

    # =========================
    # 4. Timeline
    # =========================
    t4_score = 0

    already_c = sum(1 for w in keywords_lib['T4']['already'] if w in text)
    within_c = sum(1 for w in keywords_lib['T4']['within_2y'] if w in text)
    mid_c = sum(1 for w in keywords_lib['T4']['between_2_5y'] if w in text)
    long_c = sum(1 for w in keywords_lib['T4']['above_5y'] if w in text)

    already_t = get_tfidf_sum(keywords_lib['T4']['already'])
    within_t = get_tfidf_sum(keywords_lib['T4']['within_2y'])
    mid_t = get_tfidf_sum(keywords_lib['T4']['between_2_5y'])
    long_t = get_tfidf_sum(keywords_lib['T4']['above_5y'])

    t4_score = (
        already_c * already_t+
        within_c *  within_t+
        mid_c *  mid_t+
        long_c *  long_t
    )

    if debug:
      print("=== T4 ===")
      print("already:", already_c, "*", already_t, "=", already_c * already_t)
      print("within_2y:", within_c, "*", within_t, "=", within_c * within_t)
      print("2_5y:", mid_c, "*", mid_t, "=", mid_c * mid_t)
      print("above_5y:", long_c, "*", long_t, "=", long_c * long_t)

    vec[semantic_classes['timeline_score']] = min(t4_score, 2.0)

    return torch.tensor(vec, dtype=torch.float)

In [ ]:
import torch
from torch.utils.data import Dataset

class ESGDataset(Dataset):
    def __init__(self, data, tokenizer, label2id, keywords_lib, semantic_classes, tfidf_matrix, word_to_idx):
        self.data = data
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.keywords_lib = keywords_lib
        self.semantic_classes = semantic_classes
        self.num_extra_features = len(semantic_classes)
        self.tfidf_matrix = tfidf_matrix
        self.word_to_idx = word_to_idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item["segmented_text"]

        # BERT Tokenization
        encoding = self.tokenizer(text, max_length=MAX_LEN, padding="max_length",
                                  truncation=True, return_tensors="pt")

        # 定義的函式提取特徵
        extra_features = extract_semantic_features(
            text,
            idx,
            self.keywords_lib,
            self.semantic_classes,
            self.tfidf_matrix,
            self.word_to_idx
        )

        sample = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "extra_features": extra_features,
        }

        # train / val 才有 labels
        has_labels = all(field in item for field in EVAL_FIELDS)

        if has_labels:

            labels = {
                field: self.label2id[field][item[field]]
                for field in EVAL_FIELDS
            }

            sample["labels"] = labels

        return sample


def collate_fn(batch):

    input_ids = torch.stack([
        b["input_ids"] for b in batch
    ])

    attention_mask = torch.stack([
        b["attention_mask"] for b in batch
    ])

    extra_features = torch.stack([
        b["extra_features"] for b in batch
    ])

    output = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "extra_features": extra_features,
    }

    # train / val 才有 labels
    if "labels" in batch[0]:

        labels = {
            field: torch.tensor([
                b["labels"][field]
                for b in batch
            ])
            for field in EVAL_FIELDS
        }

        output["labels"] = labels

    return output

In [ ]:
# 示範 Tokenizer 的效果
print("Tokenizer 示範")
print("=" * 50)

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
print(f"載入 {MODEL_NAME} tokenizer")

sample_text = train_data[0]["segmented_text"][:50]
print(f"\n原始文字：{sample_text}")

tokens = tokenizer.tokenize(train_data[0]["segmented_text"])
print(f"\nTokenize 後：{tokens}")

encoding = tokenizer(sample_text, max_length=20, padding="max_length", truncation=True)
print(f"\nInput IDs (前20個)：{encoding['input_ids']}")
print(f"Attention Mask：{encoding['attention_mask']}")

features = extract_semantic_features(
    text=train_data[0]["segmented_text"],
    text_idx=0,
    keywords_lib=KEYWORDS_LIB,
    semantic_classes=SEMANTIC_CLASSES,
    tfidf_matrix=tfidf_train,
    word_to_idx=word_to_idx,
    debug=True
)

print(features)



In [ ]:
# 建立 Dataset 實例
train_dataset = ESGDataset(
    train_data, tokenizer, label2id,
    KEYWORDS_LIB, SEMANTIC_CLASSES,
    tfidf_train, word_to_idx
)

val_dataset = ESGDataset(
    val_data, tokenizer, label2id,
    KEYWORDS_LIB, SEMANTIC_CLASSES,
    tfidf_val, word_to_idx
)


# 建立 DataLoader（負責批次載入資料）
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,          # 訓練時打亂順序
    collate_fn=collate_fn
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,         # 驗證時不打亂
    collate_fn=collate_fn
)

print(f"DataLoader 建立完成！")
print(f"   訓練集 Batch 數: {len(train_loader)}")
print(f"   驗證集 Batch 數: {len(val_loader)}")



In [ ]:
class MultiTaskBert(nn.Module):
    def __init__(self, num_labels_dict, extra_feat_dim=4):
        super().__init__()
        self.bert = BertModel.from_pretrained(MODEL_NAME)
        # 計算總輸入維度：BERT(768) + 專家特徵(4) = 772
        self.combined_dim = 768 + extra_feat_dim

        # 修改所有的分類器，使其接收 772 維
        self.classifier_promise = nn.Linear(self.combined_dim, num_labels_dict['promise_status'])
        self.classifier_evidence = nn.Linear(self.combined_dim, num_labels_dict['evidence_status'])
        self.classifier_quality = nn.Linear(self.combined_dim, num_labels_dict['evidence_quality'])
        self.classifier_timeline = nn.Linear(self.combined_dim, num_labels_dict['verification_timeline'])

        self.dropout = nn.Dropout(0.2)

    def forward(self, input_ids, attention_mask, extra_features):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output

        extra_features = F.normalize(extra_features, dim=1)

        # 維度[batch_size, 772]
        combined_features = torch.cat((pooled_output, extra_features), dim=1)

        # 傳入 772 維到 Linear(772, n)
        logits = {
            'promise_status': self.classifier_promise(combined_features),
            'evidence_status': self.classifier_evidence(combined_features),
            'evidence_quality': self.classifier_quality(combined_features),
            'verification_timeline': self.classifier_timeline(combined_features)
        }
        return {'logits': logits, 'embeddings': pooled_output}

# 初始化模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiTaskBert(num_labels).to(device)

print("模型維度為 BERT(768) + 專家特徵(4) = 772！")

In [ ]:


class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha  # 各類別的權重
        self.gamma = gamma  # 難度調節因子
        self.reduction = reduction

    def forward(self, inputs, targets):
        # inputs: [batch_size, num_classes] (模型輸出的 Logits)
        # targets: [batch_size] (正確標籤)

        # 1. 計算交叉熵 (不進行平均)
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')

        # 2. 計算 pt (預測正確類別的機率)
        pt = torch.exp(-ce_loss)

        # 3. 計算 Focal Loss: (1-pt)^gamma * ce_loss
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def train_one_epoch(model, dataloader, optimizer, scheduler, device, field_weights=None, alpha=5, lambda_con=0.1):
    model.train()
    total_loss = 0

    # 1. 初始化損失函數
    # 對比損失：使用 CosineEmbeddingLoss，margin=0.5 防止過度推離
    contrastive_fn = torch.nn.CosineEmbeddingLoss(margin=0.5).to(device)

    criterions = {}
    for field in EVAL_FIELDS:
        weight = field_weights[field].to(device) if field_weights else None
        criterions[field] = FocalLoss(alpha=weight, gamma=2.0).to(device)

    for step, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        extra_features = batch["extra_features"].to(device)

        # 2. R-Drop: 兩次 Forward 獲取不同 Dropout 的 Logits 與 Embeddings
        # 確保模型 forward 回傳 dict: {'logits': ..., 'embeddings': ...}
        out1 = model(input_ids, attention_mask, extra_features)
        out2 = model(input_ids, attention_mask, extra_features)

        logits1, emb1 = out1['logits'], out1['embeddings']
        logits2, emb2 = out2['logits'], out2['embeddings']

        batch_loss = 0
        for field in EVAL_FIELDS:
            target = batch["labels"][field].to(device)

            # (A) 分類 Loss: 兩次預測的平均 Focal Loss
            ce_loss = (criterions[field](logits1[field], target) +
                       criterions[field](logits2[field], target)) / 2

            # (B) KL 散度 (R-Drop): 強制預測一致性
            p = F.log_softmax(logits1[field], dim=-1)
            q = F.log_softmax(logits2[field], dim=-1)
            p_tec = F.softmax(logits1[field], dim=-1)
            q_tec = F.softmax(logits2[field], dim=-1)
            kl_loss = (F.kl_div(p, q_tec, reduction='batchmean') +
                       F.kl_div(q, p_tec, reduction='batchmean')) / 2

            # (C) 對比式學習 Loss: 區分相似語義
            con_loss = 0
            # 排除 Ignore Index (N/A 通常設為 -100)
            mask = (target != -100)
            if mask.sum() > 1:
                v_emb = emb1[mask]  # 取第一遍的 embedding 進行對比
                v_target = target[mask]

                # 滾動對比法 (Rolling Contrast): 將 Batch 內的樣本錯位配對
                # 這是在顯存有限下最推薦的對比方式
                v_emb_shifted = torch.roll(v_emb, shifts=1, dims=0)
                v_target_shifted = torch.roll(v_target, shifts=1, dims=0)

                # 標籤相同為 1, 不同為 -1
                y_con = torch.where(v_target == v_target_shifted, 1, -1).to(device)
                con_loss = contrastive_fn(v_emb, v_emb_shifted, y_con)

            # (D) 總合加權：分類 + R-Drop + 對比式學習
            batch_loss += (ce_loss + alpha * kl_loss + lambda_con * con_loss) * FIELD_WEIGHTS[field]

        # 3. 反向傳播與梯度裁剪
        optimizer.zero_grad()
        batch_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += batch_loss.item()

        if (step + 1) % 50 == 0:
            print(f"  Step {step+1}/{len(dataloader)}, Loss: {batch_loss.item():.4f}")

    return total_loss / len(dataloader)

print("訓練函數定義完成！")

In [ ]:
def predict_with_confidence(model, dataloader, device, id2label):
    """
    對資料集進行預測，同時輸出預測標籤與其對應的信心度 (Confidence Score)，
    並強制執行級聯後處理規則。
    """
    model.eval()
    predictions = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            extra_features = batch["extra_features"].to(device)

            # --- 解包模型回傳的 dict ---
            output = model(input_ids, attention_mask, extra_features)
            logits = output['logits']  # 取得各個任務的分類 Logits

            batch_size = input_ids.size(0)
            for i in range(batch_size):
                pred = {}

                # 取得每個欄位的原始預測結果與信心度
                for field in EVAL_FIELDS:
                    # 1. 取得該樣本在該任務的 logit 向量，形狀為 [num_classes]
                    field_logits = logits[field][i]

                    # 2. 通過 Softmax 將 Logits 轉成機率值 (0.0 ~ 1.0)
                    probs = torch.softmax(field_logits, dim=0)

                    # 3. 找出最大機率的 index 作為預測 ID，並將其機率作為信心度
                    pred_id = probs.argmax().item()
                    confidence = probs[pred_id].item() # 轉為 Python float

                    # 4. 存入預測結構
                    pred[field] = id2label[field][pred_id]
                    pred[f"{field}_confidence"] = round(confidence, 3)

                # --- 執行級聯規則修正 (後處理) ---
                # 級聯修正時，若被強制歸類為空字串，可以將其信心度設為 1.0 或保持原狀
                if pred['promise_status'] == 'No':
                    pred['verification_timeline'] = 'N/A'
                    pred['evidence_status'] = 'N/A'
                    pred['evidence_quality'] = 'N/A'
                    # 強制修正的欄位信心度同步調整
                    pred['verification_timeline_confidence'] = pred['promise_status_confidence']
                    pred['evidence_status_confidence'] = pred['promise_status_confidence']
                    pred['evidence_quality_confidence'] = pred['promise_status_confidence']

                elif pred['evidence_status'] == 'No':
                    pred['evidence_quality'] = 'N/A'
                    pred['evidence_quality_confidence'] = pred['evidence_status_confidence']

                predictions.append(pred)

    return predictions

print("預測函數定義完成！")

In [ ]:
def evaluate_hybrid(gt_data, pred_data):
    """
    混合評估函數：加權 Macro F1

    Args:
        gt_data: Ground truth 資料列表
        pred_data: 預測結果列表（順序需與 gt_data 一致）

    Returns:
        results: 各欄位的評估結果 + 最終加權分數
    """
    assert len(gt_data) == len(pred_data), \
        f"筆數不符：gt={len(gt_data)}, pred={len(pred_data)}"

    results = {}
    weighted_score = 0.0

    for field, labels in EVAL_FIELDS.items():
        y_true = [str(item[field]).strip() for item in gt_data]
        y_pred = [str(item[field]).strip() for item in pred_data]

        clean_labels = [str(l).strip() for l in labels]
        # Macro F1: 每個類別分別計算 F1，再取平均（不考慮類別數量）
        macro_f1 = f1_score(y_true, y_pred, labels=clean_labels, average="macro", zero_division=0)
        # Micro F1: 全部類別合併計算（考慮類別數量）
        micro_f1 = f1_score(y_true, y_pred, labels=labels, average="micro", zero_division=0)
        report   = classification_report(y_true, y_pred, labels=labels, zero_division=0)

        weight = FIELD_WEIGHTS.get(field, 0)
        weighted_score += macro_f1 * weight

        results[field] = {
            "macro_f1": macro_f1,
            "micro_f1": micro_f1,
            "report": report,
            "weight": weight
        }

    results["final_weighted_score"] = weighted_score
    return results


print(" 評估函數定義完成！")
print("\n Macro F1 vs Micro F1 說明：")
print("   Macro F1: 各類別同等重要，不受樣本數影響 → 用於評估少數類別")
print("   Micro F1: 樣本數多的類別影響較大 → 整體準確率指標")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

field_loss_weights = {}

for field in EVAL_FIELDS:
    # 1. 取得該欄位在訓練集中的所有實際標籤
    y_train = [item[field] for item in train_data]

    # 2. 找出「實際存在」於 y_train 中的類別

    present_classes = np.unique(y_train)

    # 3. 計算這些存在類別的權重
    weights = compute_class_weight(
        class_weight='balanced',
        classes=present_classes,
        y=np.array(y_train)
    )

    # 4. 建立一個對應完整 EVAL_FIELDS 的權重 Tensor（預設權重為 1.0）
    full_weights = np.ones(len(EVAL_FIELDS[field]))
    for i, cls_name in enumerate(EVAL_FIELDS[field]):
        if cls_name in present_classes:
            # 找到該類別在 compute_class_weight 結果中的位置
            idx = np.where(present_classes == cls_name)[0][0]
            full_weights[i] = weights[idx]

    # 5. 轉為 Tensor
    field_loss_weights[field] = torch.tensor(full_weights, dtype=torch.float).to(device)

    print(f"✅ {field} 權重已對齊: {dict(zip(EVAL_FIELDS[field], full_weights))}")

In [ ]:
from transformers import get_cosine_schedule_with_warmup

# 設定優化器與學習率排程器
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

total_steps   = len(train_loader) * EPOCHS
warmup_steps  = int(0.1 * total_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f"訓練設定：")
print(f"   Total steps: {total_steps}")
print(f"   Warmup steps: {warmup_steps} (前 10%)")
print(f"   Learning rate: {LR}")
print(f"   Optimizer: AdamW")

In [ ]:
# ============================================================
# 主訓練迴圈
# ============================================================
MODEL_SAVE_PATH = "best_model.pt"

best_score = 0.0
history = {"loss": [], "weighted_score": []}

print("開始訓練！")
print("=" * 60)

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 40)

    # 訓練
    avg_loss = train_one_epoch(model, train_loader, optimizer, scheduler, device, field_loss_weights, lambda_con=Lambda)
    history["loss"].append(avg_loss)
    print(f"  平均 Loss: {avg_loss:.4f}")

    # 在驗證集上評估
    preds   = predict_with_confidence(model, val_loader, device, id2label)
    results = evaluate_hybrid(val_data, preds)

    current_score = results["final_weighted_score"]
    history["weighted_score"].append(current_score)

    print(f"\n  加權分數: {current_score:.5f}")
    for field in EVAL_FIELDS:
        r = results[field]
        print(f"     {field}: Macro F1={r['macro_f1']:.4f} (權重={r['weight']})")

    # 儲存最佳模型
    if current_score > best_score:
        best_score = current_score
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"\n 最佳模型已儲存！分數: {best_score:.5f}")

print(f"\n{'=' * 60}")
print(f"訓練完成！最佳加權分數: {best_score:.5f}")

In [ ]:
# 繪製訓練曲線
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, EPOCHS + 1)

axes[0].plot(epochs_range, history["loss"], 'b-o', linewidth=2, markersize=8)
axes[0].set_title("Training Loss", fontsize=14, fontweight='bold')
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(epochs_range)

axes[1].plot(epochs_range, history["weighted_score"], 'g-o', linewidth=2, markersize=8)
axes[1].set_title("Validation Weighted Score", fontsize=14, fontweight='bold')
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Weighted F1 Score")
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(epochs_range)

# 標記最佳點
best_epoch = history["weighted_score"].index(max(history["weighted_score"])) + 1
axes[1].axvline(best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best epoch={best_epoch}')
axes[1].legend()

plt.tight_layout()
plt.savefig("training_curve.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import json
import pandas as pd

# 載入最佳模型
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
print(f"已載入最佳模型")

# 使用含有信心度的新預測函數
final_preds = predict_with_confidence(model, val_loader, device, id2label)

OUTPUT_JSON = "bert_prediction.json"
OUTPUT_CSV = "bert_prediction.csv"

output_data = []

for orig, pred in zip(val_data, final_preds):

    item = {
        "id": orig["id"],
        # 預測結果
        "promise_status": pred["promise_status"],
        "verification_timeline": pred["verification_timeline"],
        "evidence_status": pred["evidence_status"],
        "evidence_quality": pred["evidence_quality"],
        # 信心度分數
        "promise_status_confidence": pred["promise_status_confidence"],
        "verification_timeline_confidence": pred["verification_timeline_confidence"],
        "evidence_status_confidence": pred["evidence_status_confidence"],
        "evidence_quality_confidence": pred["evidence_quality_confidence"]
    }

    output_data.append(item)

# ========= 儲存 JSON =========
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"✅ JSON 已儲存至: {OUTPUT_JSON}")

# ========= 儲存 CSV =========
df = pd.DataFrame(output_data)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"✅ CSV 已儲存至: {OUTPUT_CSV}")

print(f"   共 {len(output_data)} 筆")

# ========= 評估 =========
final_results = evaluate_hybrid(val_data, final_preds)

print(f"\n{'=' * 60}")
print(f"最終評估結果")
print(f"{'=' * 60}")

for field in EVAL_FIELDS:
    r = final_results[field]

    print(f"\n--- {field} (權重: {r['weight']}) ---")
    print(r["report"])
    print(f"  Macro F1: {r['macro_f1']:.4f}")
    print(f"  Micro F1: {r['micro_f1']:.4f}")

print(f"\n{'=' * 60}")
print(f"最終加權分數: {final_results['final_weighted_score']:.5f}")
print(f"{'=' * 60}")

In [ ]:
# 視覺化各欄位的 F1 分數
fields = list(EVAL_FIELDS.keys())
macro_f1s = [final_results[f]["macro_f1"] for f in fields]
baseline = [0.728, 0.461, 0.596, 0.443]
weights = [0.20, 0.15, 0.30, 0.35]
baseline_weighted = sum(b * w for b, w in zip(baseline, weights))
x = range(len(fields))
width = 0.3

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar([i + width/2 for i in x], baseline, width, label='baseline',  color='coral',     alpha=0.8)
bars2 = ax.bar([i - width/2 for i in x], macro_f1s, width, label='bert', color='steelblue', alpha=0.8)

# 加上分數標籤
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)

ax.set_xlabel("任務欄位")
ax.set_ylabel("F1 分數")

title_text = (
    f"baseline 最終加權分數: {baseline_weighted:.4f}\n"
    f"BERT 最終加權分數: {final_results['final_weighted_score']:.4f}"
)
ax.set_title(title_text, fontsize=14, fontweight='bold')
ax.set_xticks(list(x))
ax.set_xticklabels([f"{f}\n(w={w})" for f, w in zip(fields, weights)], fontsize=9)
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

ax.legend()

plt.tight_layout()
plt.savefig("f1_scores.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# TEST PREDICTION
# ============================================================

PUBLIC_TEST_FILE = "public_test.json"
PRIVATE_TEST_FILE = "private_test.json"

# ------------------------------------------------
# 載入資料
# ------------------------------------------------

public_test_data = load_json(PUBLIC_TEST_FILE)
private_test_data = load_json(PRIVATE_TEST_FILE)

# ------------------------------------------------
# 前處理（斷詞）
# ------------------------------------------------

print("正在處理 Public Test...")
public_test_data = process_data_segmentation(public_test_data)

print("正在處理 Private Test...")
private_test_data = process_data_segmentation(private_test_data)

# ------------------------------------------------
# TF-IDF transform
# ------------------------------------------------

public_test_texts = [
    item["segmented_text"]
    for item in public_test_data
]

private_test_texts = [
    item["segmented_text"]
    for item in private_test_data
]

tfidf_public_test = tfidf_vec.transform(public_test_texts)
tfidf_private_test = tfidf_vec.transform(private_test_texts)

# ------------------------------------------------
# Dataset
# ------------------------------------------------

public_test_dataset = ESGDataset(
    public_test_data,
    tokenizer,
    label2id,
    KEYWORDS_LIB,
    SEMANTIC_CLASSES,
    tfidf_public_test,
    word_to_idx
)

private_test_dataset = ESGDataset(
    private_test_data,
    tokenizer,
    label2id,
    KEYWORDS_LIB,
    SEMANTIC_CLASSES,
    tfidf_private_test,
    word_to_idx
)

# ------------------------------------------------
# DataLoader
# ------------------------------------------------

public_test_loader = DataLoader(
    public_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

private_test_loader = DataLoader(
    private_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

# ------------------------------------------------
# 載入最佳模型
# ------------------------------------------------

model.load_state_dict(
    torch.load(MODEL_SAVE_PATH, map_location=device)
)

model.eval()

print("✅ 最佳模型載入完成")

# ------------------------------------------------
# 預測
# ------------------------------------------------

public_preds = predict_with_confidence(
    model,
    public_test_loader,
    device,
    id2label
)

private_preds = predict_with_confidence(
    model,
    private_test_loader,
    device,
    id2label
)

print("✅ 測試集預測完成")

# ============================================================
# 輸出 Public JSON
# ============================================================

public_output = []

for orig, pred in zip(public_test_data, public_preds):

    item = {
        "id": orig["id"],
        "promise_status": pred["promise_status"],
        "verification_timeline": pred["verification_timeline"],
        "evidence_status": pred["evidence_status"],
        "evidence_quality": pred["evidence_quality"]
    }

    public_output.append(item)

with open("public_prediction.json", "w", encoding="utf-8") as f:
    json.dump(public_output, f, ensure_ascii=False, indent=2)

print("✅ public_prediction.json 已輸出")

# ============================================================
# 輸出 Private JSON
# ============================================================

private_output = []

for orig, pred in zip(private_test_data, private_preds):

    item = {
        "id": orig["id"],
        "promise_status": pred["promise_status"],
        "verification_timeline": pred["verification_timeline"],
        "evidence_status": pred["evidence_status"],
        "evidence_quality": pred["evidence_quality"]
    }

    private_output.append(item)

with open("private_prediction.json", "w", encoding="utf-8") as f:
    json.dump(private_output, f, ensure_ascii=False, indent=2)

print("✅ private_prediction.json 已輸出")